https://www.census.gov/data/developers/data-sets/qwi.html?

In [1]:
import pandas as pd
import requests
from dotenv import load_dotenv
load_dotenv()
import os

In [2]:
# Codebook

vars_qwi = pd.read_csv("../Data/Census Codebooks/variables_qwi.csv")
vars_industry = pd.read_csv("../Data/Census Codebooks/label_industry.csv")
naics_codes = pd.read_csv("../Data/Census Codebooks/NAICS.csv")
agegrp_codes = pd.read_csv("../Data/Census Codebooks/label_agegrp.csv")

In [3]:
# Census Query with Generic Retry Logic, splitting each year to a separate call

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()
session.mount(
    "https://api.census.gov/",
    HTTPAdapter(max_retries=Retry(
        total=3,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )),
)

frames = []
for year in range(2020, 2027):
    response = session.get(
        "https://api.census.gov/data/timeseries/qwi/sa",
                        params={
                        "get": "Emp,HirN,Sep,industry", #Dropped Payroll since it is returning NA for the sa endpoint ---
                        "for": "state:08",
                        "time": "from 2020-Q1 to 2026-Q4",
                        "ind_level" : "S",
                        "sex": "0", # 1,2
                        "agegrp": ["A01", "A02", "A03", "A04", "A05", "A06", "A07", "A08"],
                        "key": os.environ["CENSUS_API_KEY"]
                        },
        timeout=(10, 120),
    )
    response.raise_for_status()

    if response.status_code == 204:
        continue  # No published data for this year.

    data = response.json()
    frames.append(pd.DataFrame(data[1:], columns=data[0]))

if not frames:
    raise ValueError("No QWI data returned for the requested years.")

df = pd.concat(frames, ignore_index=True)

In [ ]:
df = pd.DataFrame(response.json()[1:], columns=response.json()[0])


df["date"] = pd.PeriodIndex(
                           df["time"], 
                           freq="Q"
                            ).to_timestamp()

In [ ]:
df = df.apply( #apply -> takes each column from df as a series
    lambda col: col # input is column, output is column
        if col.name in ['time',
                        'date',
                        'industry',
                        'ind_level',
                        'agegrp']
        else pd.to_numeric(col, errors='coerce'),
        axis=0
            )

In [ ]:
# Adding Industry Names to DF with codes only

df = df.merge(
    naics_codes[['NAICS','Sector']],
    left_on = 'industry',
    right_on = 'NAICS',
    how = 'left',
    validate = 'm:1'
)

In [ ]:
# Adding age group ranges

df = df.merge(
    agegrp_codes[['agegrp','label']],
    left_on = 'agegrp',
    right_on = 'agegrp',
    how = 'left',
    validate = 'm:1'
)

df = df.rename(columns={'label':'age'})

In [ ]:
# 2 digit NAICS -> 11 Groupings

df['industry'] = df['NAICS'].map(
            {
            ('11', '21')                   : 'Natural resources and mining',
            '23'                           : 'Construction',
            '31–33'                        : 'Manufacturing',
            ('42', '44–45', '48–49', '22') : 'Trade, transportation, and utilities',
            '51'                           : 'Information',
            ('52', '53')                   : 'Financial activities',
            ('54', '55', '56')             : 'Professional and business services',
            ('61', '62')                   : 'Education and health services',
            ('71', '72')                   : 'Leisure and hospitality',
            '81'                           : 'Other services',
            '92'                           : 'Public administration'
            }
)


In [ ]:
import altair as alt


plot = alt.Chart(df).mark_area().encode(
    x='date:T',
    y='sum(Emp):Q',
    color="industry:N",
    tooltip = ['industry:N', 'sum(Emp):Q']
)

In [ ]:
plot.properties(width=750, height=400).interactive() + plot.mark_line()

In [ ]:
plot = alt.Chart(df).mark_line().encode(
        x='date:T',
        y='Emp',
        color="age:N",
        column=('Sector:N'),
        tooltip= ['age:N', 'sum(Emp):Q']
) 

In [ ]:
plot.interactive()

In [ ]:
# Summary Stats and General Trends by Age group for Colorado